# Lab 10-02 — Iterative multi-hop retrieval loop (HotpotQA)

**Track 10 · Agentic RAG** — single-shot retrieval fails on questions whose answer lives in two separate passages; a multi-hop loop turns retrieval into an iterative, model-driven process.

This notebook is **self-contained**: it imports `langchain-ollama` and `langchain-huggingface` directly — no repo component library. Every block of the pipeline is built right here: the BGE embedder (`HuggingFaceEmbeddings`), the cosine ranker, the hop loop, and the LLM's next-search-need decision all appear as plain code in the cells below, which is exactly how the shared components in `src/` work underneath.

```text
HotpotQA question (10 candidate paragraphs)
  -> BGE embeddings (HuggingFaceEmbeddings, local, cpu)
  -> hop 0: retrieve top-2 with the raw question
  -> each hop: LLM rewrites the missing-fact sub-query (json_object)
  -> retrieve top-2 unseen paragraphs by cosine
  -> done? -> LLM generates the final answer from accumulated evidence
  -> verification gate (termination, evidence, non-empty answers)
```

Each HotpotQA question carries 10 candidate paragraphs (2 gold + 8 distractors). We index those 10 paragraphs per question with local BGE embeddings and run a small loop:

* **hop 0** retrieves top-2 with the raw question (guarantees a first hit),
* each later hop asks the LLM for the next search need (a rewritten sub-query) given the question + evidence so far, then retrieves top-2 unseen paragraphs by cosine similarity,
* when the model reports enough evidence (`done`) or the hop budget runs out, the LLM generates the final answer from all accumulated evidence.

The gate is deliberately structural and tolerant: a local 7B model may retrieve the wrong paragraphs and still pass — what the lab verifies is that the loop terminates, accumulates evidence, and produces a non-empty answer.


## Setup

Two prerequisites must hold before this notebook will run:

- **Ollama serving `qwen2.5-coder:7b` at `localhost:11434`** — the local LLM behind both the hop-decision (`json_object`) and the final answer (wired up inline with `langchain-ollama`'s `ChatOllama`, `base_url="http://localhost:11434"`). Fully local: no API key, no quota.
- **HotpotQA on disk** — `Data/corpus/hotpotqa/hotpot_dev_distractor_v1.json` (the distractor dev split: 10 context paragraphs per question), already fetched by the repo's manifest-verified fetchers.

No repo imports are needed: everything this notebook uses comes from `langchain-ollama` and `langchain-huggingface`. The imports cell walks up to the repo root and cd's into it, because a notebook has no `__file__` — so every `Data/...` path resolves exactly like the lab script. Unlike the Curriculum notebook, there is no `sys.path` trick: nothing is imported from `src/`.

The next cell installs the notebook-specific dependencies (a no-op if you already ran `pip install -r requirements.txt`).


In [ ]:
# Lab-specific dependencies (already in requirements.txt — the install
# below is a no-op if you have run `pip install -r requirements.txt`).
#   langchain-ollama      -> ChatOllama, the local LLM backend (qwen2.5-coder:7b)
#   langchain-huggingface -> HuggingFaceEmbeddings, the local BGE embedder
%pip install -q langchain-ollama langchain-huggingface


In [ ]:
# Bootstrap: stdlib imports + repo-root walk (no sys.path tricks).
from __future__ import annotations

import json
import os
import sys
import time
from pathlib import Path

# LangChain — the only libraries this notebook needs. Nothing is imported
# from the repo's src/ component library.
from langchain_huggingface import HuggingFaceEmbeddings  # noqa: E402
from langchain_ollama import ChatOllama  # noqa: E402

# A notebook has no __file__, so walk up from the cwd to the repo root and
# cd into it — Data/... paths then resolve exactly like the lab script.
REPO_ROOT = Path.cwd()
for _candidate in [Path.cwd(), *Path.cwd().parents]:
    if (_candidate / "src" / "curriculum").is_dir() and (_candidate / "NoteBooks").is_dir():
        REPO_ROOT = _candidate
        break
os.chdir(REPO_ROOT)


## 1. Configuration

Everything that keeps this lab fast but still meaningful is a named constant. `N_QUESTIONS = 3` bounds the demo to three HotpotQA questions (each costs ~3-4 LLM calls); `MAX_HOPS = 3` is the retrieval-hop budget per question (hop 0 always uses the raw question); `RETRIEVE_K = 2` is how many unseen paragraphs each hop pulls. BGE runs on CPU because Ollama holds most of the GPU VRAM.


In [ ]:
# --------------------------------------------------------------------------
# 1. Configuration
# --------------------------------------------------------------------------
HOTPOT_PATH = Path("Data/corpus/hotpotqa/hotpot_dev_distractor_v1.json")
N_QUESTIONS = 3  # each question costs ~3-4 LLM calls; keep the lab fast
MAX_HOPS = 3  # retrieval hops per question (hop 0 uses the raw question)
RETRIEVE_K = 2  # paragraphs pulled per hop
BGE_MODEL_NAME = "BAAI/bge-base-en-v1.5"
BGE_DEVICE = "cpu"  # shared GPU: Ollama holds most of VRAM


## 2. Load — first N HotpotQA questions (each has 10 context paragraphs)

`load_questions` reads the first `n` records of the dev split, each with a `question`, a `context` of 10 `(title, sentences)` paragraphs, a `supporting_facts` list (the gold evidence), and an `answer`. `question_passages` flattens each paragraph's sentences into one text; `gold_paragraphs` collects the titles HotpotQA marks as required evidence; `answer_contains` is a normalized substring check used later to report (not gate) whether the gold answer ended up inside the final answer.


In [ ]:
# --------------------------------------------------------------------------
# 2. Load — first N HotpotQA questions (each has 10 context paragraphs)
# --------------------------------------------------------------------------
def load_questions(path: Path, n: int) -> list[dict]:
    """Return the first ``n`` questions from the dev set."""
    with open(path) as f:
        records = json.load(f)
    return records[:n]


def question_passages(rec: dict) -> tuple[list[str], list[str]]:
    """Return (passage_texts, paragraph_titles) for the 10 context paragraphs."""
    texts: list[str] = []
    titles: list[str] = []
    for title, sentences in rec["context"]:
        titles.append(title)
        texts.append(" ".join(sentences))
    return texts, titles


def gold_paragraphs(rec: dict) -> set[str]:
    """The paragraph titles HotpotQA marks as required evidence."""
    return {title for title, _ in rec["supporting_facts"]}


def answer_contains(gold: str, answer: str) -> bool:
    """Normalized substring check: is the gold answer inside the answer?"""
    return gold.strip().lower() in answer.strip().lower()


## 3. Experiment — iterative retrieve-then-refine loop per question

The whole loop is built inline. `_OllamaLLM` is a hand-rolled stand-in for `src/llms/ollama.py`: it wraps `ChatOllama` (model `qwen2.5-coder:7b`, `temperature=0.0`, `base_url="http://localhost:11434"`) and exposes the same `invoke(prompt) -> str` / `json_object(prompt) -> dict` contract (code-fence stripping + JSON retries); `_bge_embedder` is the hand-rolled stand-in for `src/embeddings/bge.py` (`HuggingFaceEmbeddings`, `normalize_embeddings=True`).

The loop itself: `_top_unseen` ranks the question's 10 paragraph vectors by cosine and returns top-k indices not yet seen; `_next_search_need` prompts the LLM for a JSON `{"done": bool, "search_need": str}` decision — `done` stops the hopping; `run_one_question` alternates retrieve → decide until the budget runs out, then asks the LLM for the final answer from the accumulated evidence. Each question's 10 paragraphs are embedded exactly once.


In [ ]:
# --------------------------------------------------------------------------
# 3. Experiment — iterative retrieve-then-refine loop per question
# --------------------------------------------------------------------------
class _OllamaLLM:
    """Inline stand-in for src/llms/ollama.py: ChatOllama + invoke/json_object.

    Wraps langchain-ollama's ChatOllama (local qwen2.5-coder:7b, fully local)
    and exposes the same contract the lab's components rely on: ``invoke``
    returns plain text, ``json_object`` strips code fences and retries JSON.
    """

    def __init__(self, model: str = "qwen2.5-coder:7b", temperature: float = 0.0,
                 base_url: str = "http://localhost:11434"):
        self.model = model
        self.temperature = temperature
        self._llm = ChatOllama(model=model, temperature=temperature, base_url=base_url)

    def invoke(self, prompt: str) -> str:
        return self._llm.invoke(prompt).content

    @staticmethod
    def _strip_code_fence(text: str) -> str:
        """Remove a surrounding markdown code fence (```json ... ```)."""
        lines = text.strip().splitlines()
        if lines and lines[0].startswith("```"):
            lines = lines[1:]
        if lines and lines[-1].strip() == "```":
            lines = lines[:-1]
        return "\n".join(lines).strip()

    def json_object(self, prompt: str, retries: int = 2) -> dict:
        """Ask the model to output ONLY a JSON object and parse it."""
        text = ""
        for attempt in range(retries + 1):
            full = prompt if attempt == 0 else prompt + (
                "\n\nRespond with ONLY valid JSON, no markdown.")
            text = self.invoke(full)
            try:
                parsed = json.loads(self._strip_code_fence(text))
                if isinstance(parsed, (dict, list)):
                    return parsed
            except (json.JSONDecodeError, ValueError):
                pass
        return {"error": f"could not parse JSON after {retries + 1} attempts",
                "raw": text}


def _bge_embedder() -> HuggingFaceEmbeddings:
    """Inline stand-in for src/embeddings/bge.py: BGE via HuggingFaceEmbeddings.

    bge models require normalized embeddings for cosine similarity, so
    ``normalize_embeddings=True`` is set exactly like the shared component.
    """
    return HuggingFaceEmbeddings(
        model_name=BGE_MODEL_NAME,
        model_kwargs={"device": BGE_DEVICE},
        encode_kwargs={"normalize_embeddings": True},  # bge needs cosine-normalized vectors
    )


def _cosine(a: list[float], b: list[float]) -> float:
    norm_a = sum(x * x for x in a) ** 0.5
    norm_b = sum(x * x for x in b) ** 0.5
    if norm_a == 0.0 or norm_b == 0.0:
        return 0.0
    return sum(x * y for x, y in zip(a, b)) / (norm_a * norm_b)


def _top_unseen(query: str, vecs: list[list[float]], embedder,
                seen: set[int], k: int) -> list[int]:
    """Cosine top-k indices over ``vecs``, skipping indices already seen."""
    query_vec = embedder.embed_documents([query])[0]
    ranked = sorted(
        range(len(vecs)),
        key=lambda i: _cosine(query_vec, vecs[i]),
        reverse=True,
    )
    return [i for i in ranked if i not in seen][:k]


def _next_search_need(llm, question: str, evidence: list[dict]) -> str | None:
    """Ask the LLM for the next sub-query; None means 'enough evidence'."""
    evidence_block = "\n".join(
        f"- {item['title']}: {item['text'][:180]}..." for item in evidence
    ) or "- (none yet)"
    prompt = (
        "You are doing multi-hop retrieval for a question.\n"
        "Given the question and the evidence gathered so far, decide the "
        "next search need.\n"
        "Rules:\n"
        '- If the evidence already suffices to answer, respond '
        '{"done": true, "search_need": ""}.\n'
        '- Otherwise respond {"done": false, "search_need": "a short, '
        'specific query for the missing fact"}.\n'
        "- Output ONLY JSON.\n\n"
        f"Question: {question}\n\n"
        f"Evidence so far:\n{evidence_block}"
    )
    result = llm.json_object(prompt)
    if not isinstance(result, dict) or "error" in result:
        return None  # cannot refine -> stop hopping
    done = result.get("done", False)
    if done is True or str(done).strip().lower() in ("true", "1", "yes"):
        return None
    need = str(result.get("search_need", result.get("sub_query", ""))).strip()
    return need or None


def run_one_question(rec: dict, llm, embedder) -> dict:
    passages, titles = question_passages(rec)
    vecs = embedder.embed_documents(passages)  # embed the 10 paragraphs once
    gold = rec["answer"]
    gold_titles = gold_paragraphs(rec)

    evidence: list[dict] = []
    hops: list[tuple[str, list[str]]] = []
    seen: set[int] = set()
    llm_calls = 0
    sub_query = rec["question"]
    for hop in range(MAX_HOPS):
        hits = _top_unseen(sub_query, vecs, embedder, seen, RETRIEVE_K)
        if not hits:
            break
        seen.update(hits)
        evidence.extend(
            {"title": titles[i], "text": passages[i]} for i in hits
        )
        hops.append((sub_query, [titles[i] for i in hits]))
        if hop == MAX_HOPS - 1:
            break
        llm_calls += 1
        next_query = _next_search_need(llm, rec["question"], evidence)
        if next_query is None:
            break
        sub_query = next_query

    evidence_block = "\n\n".join(
        f"[{item['title']}] {item['text']}" for item in evidence
    )
    llm_calls += 1
    answer = llm.invoke(
        "Answer the multi-hop question using ONLY the evidence passages "
        "below. If the evidence is insufficient, say what is missing.\n\n"
        f"Evidence:\n{evidence_block}\n\n"
        f"Question: {rec['question']}\n\n"
        "Answer in one or two sentences."
    ).strip()

    return {
        "question": rec["question"],
        "hops": hops,
        "evidence_titles": [titles[i] for i in seen],
        "answer": answer,
        "gold": gold,
        "gold_titles": sorted(gold_titles),
        "answer_contains_gold": answer_contains(gold, answer),
        "llm_calls": llm_calls,
    }


def run_experiment() -> dict:
    questions = load_questions(HOTPOT_PATH, N_QUESTIONS)
    llm = _OllamaLLM()  # local qwen2.5-coder:7b via ChatOllama
    embedder = _bge_embedder()  # inline BGE via HuggingFaceEmbeddings

    t0 = time.perf_counter()
    rows = [run_one_question(rec, llm, embedder) for rec in questions]
    total_s = time.perf_counter() - t0

    return {
        "rows": rows,
        "total_s": total_s,
        "agg": {
            "questions": len(rows),
            "total_llm_calls": sum(row["llm_calls"] for row in rows),
            "hits_gold": sum(
                bool(set(row["evidence_titles"]) & set(row["gold_titles"]))
                for row in rows
            ),
        },
    }


## 4. Demo — print the artifact

The demo prints the artifact per question: every hop with its sub-query and the paragraph titles it pulled, the accumulated evidence, the final answer, the gold answer and its paragraphs, and whether the gold answer ended up in the final answer; then the aggregates (total LLM calls, how many questions' evidence touched a gold paragraph) and the takeaway: multi-hop retrieval is a loop, not a single query — each hop re-embeds a model-refined sub-query and grows the evidence, and the LLM decides when to stop (`done`), turning retrieval into an agentic, budget-bounded process.


In [ ]:
# --------------------------------------------------------------------------
# 4. Demo — print the artifact
# --------------------------------------------------------------------------
def print_demo(exp: dict) -> None:
    print("=" * 66)
    print("Lab 10-02 — Iterative multi-hop retrieval loop (HotpotQA)")
    print(f"{exp['agg']['questions']} questions in {exp['total_s']:.1f}s, "
          f"{exp['agg']['total_llm_calls']} LLM calls")
    print("=" * 66)

    for i, row in enumerate(exp["rows"], start=1):
        print(f"\nQ{i}: {row['question'][:90]}")
        for hop, (sub_query, hit_titles) in enumerate(row["hops"], start=1):
            print(f"    hop {hop}: query {sub_query!r} -> {hit_titles}")
        print(f"    evidence    : {row['evidence_titles'][:4]}")
        print(f"    final answer: {row['answer'][:120]}")
        print(f"    gold answer : {row['gold']}  "
              f"(gold paras: {', '.join(row['gold_titles'])[:70]})")
        print(f"    gold in answer: {row['answer_contains_gold']}")

    a = exp["agg"]
    print(f"\n[5] Aggregates over {a['questions']} questions")
    print(f"    total LLM calls        : {a['total_llm_calls']}")
    print(f"    questions whose evidence touched a gold paragraph: "
          f"{a['hits_gold']}/{a['questions']}")

    print(f"\n[6] Takeaway")
    print("    Multi-hop retrieval is a loop, not a single query: each hop")
    print("    re-embeds a model-refined sub-query and grows the evidence.")
    print("    The LLM decides when to stop (done) — turning retrieval into")
    print("    an agentic, budget-bounded process. Answer quality is not")
    print("    gated here; termination, evidence accumulation, and a non-")
    print("    empty answer are.")


## 5. Verification gate

The gate is deliberately structural and tolerant — a local 7B model may retrieve the wrong paragraphs and still pass. It checks: exactly `N_QUESTIONS` questions processed, every question has a non-empty hop trace, every run terminates within `MAX_HOPS` hops, every question accumulated at least one unique retrieved paragraph, and every final answer is non-empty.


In [ ]:
# --------------------------------------------------------------------------
# 5. Verification gate — run ``python <lab> --verify`` from the repo root
# --------------------------------------------------------------------------
def verify_gate(exp: dict) -> int:
    checks: list[tuple[str, bool]] = []
    rows = exp["rows"]

    checks.append((f"exactly {N_QUESTIONS} questions processed",
                   exp["agg"]["questions"] == N_QUESTIONS))
    checks.append(("every question has a non-empty hop trace",
                   all(row["hops"] for row in rows)))
    checks.append((f"every question terminates within {MAX_HOPS} hops",
                   all(len(row["hops"]) <= MAX_HOPS for row in rows)))
    checks.append(("every question accumulated >= 1 unique retrieved paragraph",
                   all(row["evidence_titles"] for row in rows)))
    checks.append(("every final answer is non-empty",
                   all(row["answer"].strip() for row in rows)))

    print("verification gate:")
    for label, ok in checks:
        print(f"  [{'PASS' if ok else 'FAIL'}] {label}")
    return 0 if all(ok for _, ok in checks) else 1


## Run the experiment

Three HotpotQA questions, each costing ~3-4 local LLM calls plus a few seconds of embedding — expect a few minutes total. No downloads, no API calls. `exp` holds everything the demo and gate need.


In [ ]:
exp = run_experiment()


### Demo — the artifact

Per question: hop-by-hop sub-queries and hits, accumulated evidence, final vs gold answer, and whether the gold answer surfaced in the final answer — then the aggregates and takeaway.


In [ ]:
print_demo(exp)


### Verification gate

Expect every check to PASS — the same gate the CI-style `--verify` run enforces. If any line shows FAIL, check that Ollama is serving `qwen2.5-coder:7b` and that the HotpotQA dev file is intact.


In [ ]:
verify_gate(exp)
